# 02 — Entrenamiento y comparación de modelos

Modelos: **Ridge**, **Decision Tree**, **Random Forest**. Opcional: **Gradient Boosting** (`USE_GRADIENT_BOOSTING = True` en configuración).

**Dataset:** [UCI](https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction)

## Split temporal (80/20)

Usamos partición **cronológica**: los primeros 80% de registros para entrenar y el 20% final para prueba.

Un `train_test_split` aleatorio mezclaría el futuro con el pasado y **sobreestimaría** el rendimiento. En series temporales lo realista es entrenar con el pasado y evaluar en periodos posteriores.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

TARGET = "Appliances"
DATE_COL = "date"
DROP_COLS = ["rv1", "rv2"]
TEST_RATIO = 0.2
RANDOM_STATE = 42
USE_GRADIENT_BOOSTING = False  # True para incluir cuarto modelo

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "raw" / "energydata_complete.csv"
FIGURES_DIR = ROOT / "reports" / "figures"
METRICS_DIR = ROOT / "reports" / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Coloca energydata_complete.csv en {DATA_PATH}")

In [ ]:
def load_and_clean(csv_path: Path) -> pd.DataFrame:
    """Carga el CSV UCI y aplica limpieza básica."""
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include="object").columns:
        if col != DATE_COL:
            df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors="coerce")
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df = df.sort_values(DATE_COL).reset_index(drop=True)
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
    df = df.drop_duplicates()
    num_cols = df.select_dtypes(include="number").columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    return df


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Variables temporales, climáticas y de rezago (solo información pasada)."""
    out = df.copy()
    dt = out[DATE_COL]

    out["hour"] = dt.dt.hour
    out["day_of_week"] = dt.dt.dayofweek
    out["month"] = dt.dt.month
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["day_sin"] = np.sin(2 * np.pi * out["day_of_week"] / 7)
    out["day_cos"] = np.cos(2 * np.pi * out["day_of_week"] / 7)

    t_cols = [f"T{i}" for i in range(1, 10) if f"T{i}" in out.columns]
    rh_cols = [f"RH_{i}" for i in range(1, 10) if f"RH_{i}" in out.columns]
    out["T_inside_mean"] = out[t_cols].mean(axis=1)
    out["RH_inside_mean"] = out[rh_cols].mean(axis=1)
    out["delta_T_out_inside"] = out["T_out"] - out["T_inside_mean"]

    out["Appliances_lag_1"] = out[TARGET].shift(1)
    out["Appliances_lag_3"] = out[TARGET].shift(3)
    out["Appliances_lag_6"] = out[TARGET].shift(6)
    past = out[TARGET].shift(1)
    out["Appliances_roll_3"] = past.rolling(3, min_periods=3).mean()
    out["Appliances_roll_6"] = past.rolling(6, min_periods=6).mean()
    out["Appliances_roll_12"] = past.rolling(12, min_periods=12).mean()
    return out


def prepare_dataset(csv_path: Path) -> pd.DataFrame:
    """Limpieza, features y drop de NaN por lag/rolling."""
    df = add_features(load_and_clean(csv_path))
    before = len(df)
    df = df.dropna().reset_index(drop=True)
    print(f"Filas tras lag/rolling: {len(df):,} (eliminadas: {before - len(df):,})")
    return df


def temporal_train_test_split(df: pd.DataFrame, test_ratio: float = TEST_RATIO):
    """Partición cronológica 80/20."""
    split_idx = int(len(df) * (1 - test_ratio))
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()


def get_xy(df: pd.DataFrame):
    y = df[TARGET]
    X = df.drop(columns=[TARGET, DATE_COL], errors="ignore")
    return X, y


def save_fig(name: str) -> None:
    path = FIGURES_DIR / name
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Figura guardada: {path}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor


def regression_metrics(y_true, y_pred) -> dict:
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_true, y_pred)
    return {"mae": mae, "mse": mse, "rmse": rmse, "r2": r2}


def make_ridge_pipeline(feature_names):
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), feature_names),
        ])),
        ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    ])


def make_tree_pipeline(estimator, feature_names):
    """Árboles: sin StandardScaler (no es necesario)."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]), feature_names),
        ])),
        ("model", estimator),
    ])

In [ ]:
df = prepare_dataset(DATA_PATH)
train_df, test_df = temporal_train_test_split(df)
X_train, y_train = get_xy(train_df)
X_test, y_test = get_xy(test_df)
feature_names = list(X_train.columns)
print(f'Train: {len(X_train):,} ({train_df[DATE_COL].min().date()} a {train_df[DATE_COL].max().date()})')
print(f'Test:  {len(X_test):,} ({test_df[DATE_COL].min().date()} a {test_df[DATE_COL].max().date()})')

### Evitar data leakage
- Rezagos (`Appliances_lag_*`) usan solo valores **anteriores**.
- Rolling usa `shift(1)` antes de la ventana.
- La variable objetivo no entra como feature contemporánea.

In [ ]:
models = {
    'Ridge': make_ridge_pipeline(feature_names),
    'Decision Tree': make_tree_pipeline(
        DecisionTreeRegressor(max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE),
        feature_names,
    ),
    'Random Forest': make_tree_pipeline(
        RandomForestRegressor(n_estimators=100, max_depth=16, min_samples_leaf=3,
                                random_state=RANDOM_STATE, n_jobs=-1),
        feature_names,
    ),
}
if USE_GRADIENT_BOOSTING:
    models['Gradient Boosting'] = make_tree_pipeline(
        GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1,
                                  random_state=RANDOM_STATE),
        feature_names,
    )

results = {}
predictions = {}
fitted = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    fitted[name] = pipe
    predictions[name] = y_pred
    results[name] = regression_metrics(y_test, y_pred)
    m = results[name]
    print(f"{name}: RMSE={m['rmse']:.2f} | R²={m['r2']:.3f}")

In [ ]:
metrics_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'model'})
metrics_df = metrics_df.sort_values('rmse').reset_index(drop=True)
metrics_df.to_csv(METRICS_DIR / 'model_metrics.csv', index=False)
metrics_df

In [ ]:
best_model = metrics_df.loc[0, 'model']
y_pred_best = predictions[best_model]

pred_out = test_df[[DATE_COL]].copy()
pred_out['y_true'] = y_test.values
pred_out['y_pred'] = y_pred_best
pred_out.to_csv(METRICS_DIR / 'predictions_best_model.csv', index=False)
print(f'Mejor modelo: {best_model}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(metrics_df['model'], metrics_df['rmse'])
axes[0].set_title('RMSE por modelo (menor es mejor)')
axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(metrics_df['model'], metrics_df['r2'])
axes[1].set_title('R² por modelo (mayor es mejor)')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
save_fig('05_metrics_comparison.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_best, alpha=0.35, s=12)
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
ax.plot(lims, lims, linestyle='--', label='Predicción perfecta')
ax.set_xlabel('Valor real (Wh)')
ax.set_ylabel('Valor predicho (Wh)')
ax.set_title(f'Real vs. Predicho — {best_model}')
ax.legend()
plt.tight_layout()
save_fig('06_actual_vs_predicted.png')
plt.show()